# Dual-T4 benchmark: SD-Turbo → SVG

**Notebook settings (right sidebar):** Accelerator = **GPU T4 x2**, Internet = **On** (needed to clone the repo and download `stabilityai/sd-turbo`; the benchmark is not the offline competition submission).

Then use *Run All*. Results land in `reports/` and are included in the notebook output. Download them and commit them to the repo.

In [ ]:
!git clone -q https://github.com/erfunm/diffusion-to-svg-pipeline.git
%cd diffusion-to-svg-pipeline
!git log -1 --oneline

## Install
`torch` is preinstalled on Kaggle and is deliberately not reinstalled here.

In [ ]:
!pip install -q -r requirements.txt pytest

## Check xFormers
xFormers must match the installed torch build. If it does not import, the benchmark still runs: the `xformers` configs are recorded with `xformers_active=False` and `attn_processor` shows what was used. Don't quote xFormers numbers from such a run.
If you want to try installing it, uncomment the next line. It must not upgrade torch: check that the torch version printed afterwards is unchanged.

In [ ]:
# !pip install -q xformers --no-deps
import torch
print("torch", torch.__version__, "CUDA", torch.version.cuda)
try:
    import xformers, xformers.ops
    print("xformers", xformers.__version__)
except Exception as e:
    print("xformers NOT available:", e)

## GPUs

In [ ]:
import torch
n = torch.cuda.device_count()
print("torch.cuda.device_count() =", n)
for i in range(n):
    print(f"cuda:{i}", torch.cuda.get_device_name(i))
assert n >= 2, "Expected 2 GPUs: set Accelerator to 'GPU T4 x2'"

In [ ]:
!nvidia-smi

## Tests (CPU-level sanity check)

In [ ]:
!python -m pytest -q

## Benchmark
Runs every configuration in `src/benchmark.py` on `prompts/benchmark_prompts.txt` with fixed per-prompt seeds.
Writes `reports/benchmark.csv`, `reports/figures/benchmark.png` and `reports/nvidia_smi_dual.txt`.

In [ ]:
!python -m src.benchmark --config configs/default.yaml

## Results

In [ ]:
import pandas as pd
from IPython.display import Image, display
df = pd.read_csv("reports/benchmark.csv")
display(df)
display(Image("reports/figures/benchmark.png"))

In [ ]:
!cat reports/nvidia_smi_dual.txt | head -60

## Constraint check on the dual-GPU outputs

In [ ]:
!python -m src.check_svgs outputs/benchmark/2gpu_fp16_xformers

## Package results for download

In [ ]:
!cd /kaggle/working && zip -qr benchmark_reports.zip diffusion-to-svg-pipeline/reports && ls -la benchmark_reports.zip